# Feedforward Pointcloud Reconstruction

This tutorial runs **VGGT-X**, **MapAnything**, and **VGGT-Omega** on keyframes extracted from a real video, then compares the resulting pointclouds side by side. All three models run feedforward inference (no optimisation loop) and write results to zarr caches that downstream notebooks — `semantic_lifting`, `localization`, and `bundle_adjustment` — depend on. **Run this notebook before any of those.**

**Prerequisite:** [Keyframe Extraction](../01_preprocessing/keyframe_extraction.ipynb) must have been executed first to populate the `images/` directory used here.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import open3d as o3d
import pyvista as pv
import torch
%matplotlib inline

# pv.set_jupyter_backend("static" if os.environ.get("PYVISTA_OFF_SCREEN") else "trame")
pv.set_jupyter_backend("trame")

from collab_splats.pointcloud.feedforward import VGGTXCreator, MapAnythingCreator, VGGTOmegaCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.pointcloud.utils import clean_pointcloud
from collab_splats.utils.visualization import (
    CAMERA_KWARGS,
    PCD_KWARGS,
    VIZ_KWARGS,
    create_camera_frustum_pyvista,
    pointcloud_to_polydata,
    visualize_splat,
)

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────

assert IMAGES.exists() and any(IMAGES.glob("*.jpg")), (
    f"No images found in {IMAGES}. Run 01_preprocessing/keyframe_extraction first."
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}  |  images: {len(list(IMAGES.glob('*.jpg')))}")


def _clean(result):
    """Filter outliers, downsample; return (pts3d, colors, conf_mean, conf_std).

    Note: conf_mean/conf_std are computed on the raw (pre-filter) confidence tensor.
    """
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(result.points)
    pcd.colors = o3d.utility.Vector3dVector(result.colors.astype(np.float64) / 255.0)
    cleaned, _ = clean_pointcloud(pcd)
    pts3d = np.asarray(cleaned.points, dtype=np.float32)
    colors = (np.asarray(cleaned.colors) * 255).astype(np.uint8)
    conf = result.confidence
    conf_mean = conf.cpu().float().mean().item() if conf is not None else float("nan")
    conf_std  = conf.cpu().float().std().item()  if conf is not None else float("nan")
    print(f"Points: {len(result.points):,} raw → {len(pts3d):,} filtered")
    print(f"Conf:   mean={conf_mean:.3f}  std={conf_std:.3f}")
    return pts3d, colors, conf_mean, conf_std

## §1 — VGGT-X Reconstruction

Loads a cached VGGT-X reconstruction from zarr if available, skipping GPU inference (~3–5 min on GPU). If no cache exists, runs VGGT-X on the keyframes and saves the result. The zarr file is chunked by frame for efficient per-frame random access by downstream notebooks (semantic_lifting, localization).

In [ ]:
_vggtx_cache = CACHE_DIR / "vggtx" / "reconstruction.zarr"
_vggtx_cache.parent.mkdir(parents=True, exist_ok=True)

if _vggtx_cache.exists():
    result_vggt = FeedforwardResult.load_zarr(_vggtx_cache)
    print(f"Loaded VGGT-X result from cache  ({result_vggt.points.shape[0]:,} pts)")
else:
    result_vggt = VGGTXCreator().run(IMAGES, device)
    result_vggt.save_zarr(_vggtx_cache)
    print(f"VGGT-X done  →  saved to {_vggtx_cache}")

## §2 — VGGT-X Post-processing and Visualisation

Removes statistical outliers and downsamples via voxel grid to produce a clean pointcloud. Confidence statistics are printed to help diagnose prediction quality.

In [ ]:
pts3d_vggt, colors_vggt, conf_vggt_mean, conf_vggt_std = _clean(result_vggt)

## §3 — VGGT-X Pointcloud Viewer

Renders the filtered VGGT-X pointcloud with camera frustums overlaid. Blue frustums show predicted camera poses.

In [ ]:
cloud_vggt = pointcloud_to_polydata(pts3d_vggt, RGB=colors_vggt)
visualize_splat(
    cloud_vggt,
    aligned_cameras=list(result_vggt.extrinsics),
    mesh_kwargs={**PCD_KWARGS, "rgb": True},
    camera_kwargs={k: v for k, v in CAMERA_KWARGS.items() if k != "color"},
    viz_kwargs=VIZ_KWARGS,
).show()

## §4 — MapAnything Reconstruction

MapAnything predicts dense depth and camera poses via cross-view consistency.
Uses `confidence_percentile` to mask unreliable pixels before unprojection.

In [ ]:
_ma_cache = CACHE_DIR / "mapanything" / "reconstruction.zarr"
_ma_cache.parent.mkdir(parents=True, exist_ok=True)

if _ma_cache.exists():
    result_ma = FeedforwardResult.load_zarr(_ma_cache)
    print(f"Loaded MapAnything result from cache  ({result_ma.points.shape[0]:,} pts)")
else:
    result_ma = MapAnythingCreator().run(IMAGES, device)
    result_ma.save_zarr(_ma_cache)
    print(f"MapAnything done  →  saved to {_ma_cache}")

## §5 — MapAnything Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as VGGT-X. Confidence metrics are collected for the side-by-side comparison table.

In [ ]:
pts3d_ma, colors_ma, conf_ma_mean, conf_ma_std = _clean(result_ma)

## §6 — MapAnything Pointcloud Viewer

Renders the filtered MapAnything pointcloud with camera frustums. Orange frustums show predicted camera poses; compare pose spread vs. VGGT-X above.

In [ ]:
cloud_ma = pointcloud_to_polydata(pts3d_ma, RGB=colors_ma)
visualize_splat(
    cloud_ma,
    aligned_cameras=list(result_ma.extrinsics),
    mesh_kwargs={**PCD_KWARGS, "rgb": True},
    camera_kwargs={k: v for k, v in CAMERA_KWARGS.items() if k != "color"},
    viz_kwargs=VIZ_KWARGS,
).show()

## §7 — VGGT-Omega Reconstruction

Loads a cached VGGT-Omega reconstruction from zarr if available, skipping GPU inference (~3–5 min on GPU). VGGT-Omega is a third feedforward method that uses a unified vision transformer for depth and pose estimation. Runs with default settings (`VGGTOmegaCreator()`).

In [ ]:
_omega_cache = CACHE_DIR / "omega" / "reconstruction.zarr"
_omega_cache.parent.mkdir(parents=True, exist_ok=True)

if _omega_cache.exists():
    result_omega = FeedforwardResult.load_zarr(_omega_cache)
    print(f"Loaded VGGT-Omega result from cache  ({result_omega.points.shape[0]:,} pts)")
else:
    result_omega = VGGTOmegaCreator().run(IMAGES, device)
    result_omega.save_zarr(_omega_cache)
    print(f"VGGT-Omega done  →  saved to {_omega_cache}")

## §8 — VGGT-Omega Post-processing and Visualisation

Applies the same outlier removal and voxel downsampling pipeline as §2/§5. Confidence metrics are collected for the three-way comparison table.

In [ ]:
pts3d_omega, colors_omega, conf_omega_mean, conf_omega_std = _clean(result_omega)

## §9 — VGGT-Omega Pointcloud Viewer

Renders the filtered VGGT-Omega pointcloud with camera frustums. Camera frustums show predicted camera poses; compare pose spread vs. VGGT-X (§3) and MapAnything (§6) above.

In [ ]:
cloud_omega = pointcloud_to_polydata(pts3d_omega, RGB=colors_omega)
visualize_splat(
    cloud_omega,
    aligned_cameras=list(result_omega.extrinsics),
    mesh_kwargs={**PCD_KWARGS, "rgb": True},
    camera_kwargs={k: v for k, v in CAMERA_KWARGS.items() if k != "color"},
    viz_kwargs=VIZ_KWARGS,
).show()

## §10 — Side-by-side Comparison

Tabulates raw vs. filtered point counts and confidence statistics for all three models. Higher confidence mean with lower std indicates more reliable depth predictions.

In [ ]:
col = 14
print(f"{'Model':<{col}} {'Pts raw':>10} {'Pts filt':>10} {'Conf mean':>10} {'Conf std':>10}")
print("-" * (col + 42))
print(f"{'VGGT-X':<{col}} {len(result_vggt.points):>10,} {len(pts3d_vggt):>10,} {conf_vggt_mean:>10.3f} {conf_vggt_std:>10.3f}")
print(f"{'MapAnything':<{col}} {len(result_ma.points):>10,} {len(pts3d_ma):>10,} {conf_ma_mean:>10.3f} {conf_ma_std:>10.3f}")
print(f"{'VGGT-Omega':<{col}} {len(result_omega.points):>10,} {len(pts3d_omega):>10,} {conf_omega_mean:>10.3f} {conf_omega_std:>10.3f}")

## §11 — Camera Pose Overlay

Overlays camera frustums from all three models in a single scene — blue for VGGT-X, orange for MapAnything, green for VGGT-Omega. Alignment across the three sets indicates consistent global pose estimation.

In [ ]:
pl = pv.Plotter()
for ext in result_vggt.extrinsics:
    pl.add_mesh(create_camera_frustum_pyvista(ext, scale=0.05), color="cornflowerblue", line_width=2)
for ext in result_ma.extrinsics:
    pl.add_mesh(create_camera_frustum_pyvista(ext, scale=0.05), color="darkorange", line_width=2)
for ext in result_omega.extrinsics:
    pl.add_mesh(create_camera_frustum_pyvista(ext, scale=0.05), color="mediumseagreen", line_width=2)
pl.camera_position = [
    VIZ_KWARGS["position"], VIZ_KWARGS["focal_point"], VIZ_KWARGS["view_up"]
]
pl.camera.azimuth = VIZ_KWARGS["azimuth"]
pl.camera.elevation = VIZ_KWARGS["elevation"]
pl.camera.Zoom(VIZ_KWARGS["zoom"])
pl.add_axes()
pl.show()